In [ ]:
# PyTorch 用来批量表示动作与奖励；DuckEnv 提供统一的仿真交互接口。
import torch
from training.sim import SKILLS, DuckEnv, hardware

In [ ]:
# 任务 walk 决定怎么走、怎么计分；硬件版本决定仿真用哪套机体与舵机参数。
# num_envs 是并行的鸭子数量；在 GPU 上一批推进，保留渲染供最后观察。
env = DuckEnv(SKILLS["walk"], hardware.load("2026-11-26-hd1910"), num_envs=16, device="cuda", render=True)

In [ ]:
# reset 给出起点：obs 是策略在部署时能用的观测，critic_obs 还含仿真额外信息。
# 张量第一维是环境编号，第二维是每只鸭子的观测分量。
obs, critic_obs = env.reset()
obs.shape, critic_obs.shape

In [ ]:
# 每行是一只鸭子，14 列是动作分量；全零表示不给默认姿态加偏移。
actions = torch.zeros(16, 14, device="cuda")
# rewards 记录这一步的即时奖励，dones 标记哪些环境的回合已结束。
obs, critic_obs, rewards, dones, info = env.step(actions)

In [ ]:
# 每只鸭子各记一份 50 步的折扣奖励和；0.99 ** t 让越远的奖励权重越小。
# 此处固定步数累加，不按 dones 把不同回合拆开。
ret = torch.zeros(16, device="cuda")
for t in range(50):
    actions = torch.randn(16, 14, device="cuda")
    obs, critic_obs, rewards, dones, info = env.step(actions)
    ret += 0.99 ** t * rewards
ret

In [ ]:
# 从初始状态再试随机动作，并逐步收集画面，用录像解释低回报从何而来。
env.reset()
frames = []
for t in range(100):
    env.step(torch.randn(16, 14, device="cuda"))
    frames.append(env.render())
import mediapy
mediapy.show_video(frames, fps=50, width=480)